<img src="icon.png" width=128/>

# Allo: Accelerator Design and Programming Language

Lecture: SEU - FPGA HLS Design

Speaker: Kai Shao

Date: 2026/06/09

# 4 · Backend Synthesis to FPGA

> **FPGA HLS Design** — Allo hands-on series (4/4)

We now close the loop the whole course has been building toward: take a verified Allo
design, **synthesize it to hardware**, and read the quality-of-results report.

The lecture's final claim was that *the future of hardware design is schedule-driven,
composable, and dataflow-oriented*, and that **regular systolic arrays — notoriously
hard to coax out of plain HLS — become natural when you can describe the spatial schedule
explicitly.** This notebook makes that concrete with the **systolic GEMM** as the running
example.

We cover:

1. From schedule to **HLS C++** (and where the pragmas come from).
2. The **systolic GEMM**: how the spatial array compiles, PE by PE.
3. **Interfaces** (AXI) for deployment.
4. **Synthesis** and reading the report (latency / II / resources).
5. Naive vs systolic — a QoR comparison.

> Most cells that *synthesize* require a Vitis HLS toolchain and take minutes; they are
> gated on `is_vitis_available()`. Code-generation cells run anywhere.

In [ ]:
import tempfile
import xml.etree.ElementTree as ET
import numpy as np

import allo.exp as allo
from allo.exp.lang.kernel import kernel
from allo.exp.lang.core import i32, f32, Stream
from allo.exp.backend.vitis.core import is_vitis_available

PART = "xcvu9p-flga2104-2-i"     # an AMD/Xilinx UltraScale+ part
print("Vitis available:", is_vitis_available())

# In VSCode/Jupyter, route Allo's logs to plain text instead of a live spinner
# widget. The spinner can render as an empty Output() in VSCode, making csim /
# synthesis look like it never starts (it is actually running underneath).
import allo.exp.logging as _allo_log
from rich.console import Console as _Console
_allo_log.console = _Console(stderr=True, force_interactive=False)

## 4.1 From schedule to HLS C++

`export("vitis").hls_code` is the synthesizable C++. The crucial point versus hand-written
HLS: **the pragmas are emitted from the schedule, not welded into the algorithm.** The
loop body is the pristine algorithm; the `#pragma`s are a *consequence* of the transform
program we attached.

In [ ]:
M = N = K = 16

@kernel
def gemm(A: f32[M, K], B: f32[K, N], C: f32[M, N]):
    for i in allo.range(M, name="i"):
        for j in allo.range(N, name="j"):
            for k in allo.range(K, name="k"):
                C[i, j] += A[i, k] * B[k, j]

s = gemm.schedule()
i, j, k = s.loops("i", "j", "k")
s.tile((i, j), factors=[4, 4])
s.pipeline(s.loop("k"), ii=1)
print(s.export("vitis").hls_code)

## 4.2 The systolic GEMM: a spatial array that compiles cleanly

Here is the **2D output-stationary systolic GEMM** again (from notebook 3), now viewed
through the *synthesis* lens. The design is an `(M+2) × (N+2)` grid of PEs:

```
        B injected from the top
        │   │   │
   A → [ ] [ ] [ ] → drain        each interior [ ] :  c += a*b
   A → [ ] [ ] [ ] → drain                          forward a → right
   A → [ ] [ ] [ ] → drain                          forward b → down
        │   │   │
        drain (bottom)
```

Each PE is a *separate* `@kernel(mapping=[P0,P1])` instance that branches on its
`get_wid` coordinate. Watch what the compiler does with it:

* **Every PE is specialized into its own function** (`systolic_2d_pe_<i>_<j>`), because
  the branch each PE takes is determined by its compile-time coordinate.
* **Idle corner PEs are pruned entirely** — they generate no hardware.

That per-PE specialization is precisely the *spatial schedule* that plain C-plus-pragmas
struggles to express.

In [ ]:
M2, N2, K2 = 2, 2, 2
P0, P1 = M2 + 2, N2 + 2

@kernel
def systolic_2d(A: f32[M2, K2], B: f32[K2, N2], C: f32[M2, N2]):
    fifo_A: Stream[f32][P0, P1]
    fifo_B: Stream[f32][P0, P1]

    @kernel(mapping=[P0, P1])
    def pe(A: f32[M2, K2], B: f32[K2, N2], C: f32[M2, N2],
           fifo_A: Stream[f32][P0, P1], fifo_B: Stream[f32][P0, P1]):
        i = allo.get_wid(0)
        j = allo.get_wid(1)
        if (i == 0 or i == M2 + 1) and (j == 0 or j == N2 + 1):
            pass
        elif j == 0:
            for k in range(K2):
                fifo_A[i, j + 1].put(A[i - 1, k])
        elif i == 0:
            for k in range(K2):
                fifo_B[i + 1, j].put(B[k, j - 1])
        elif i == M2 + 1:
            for k in range(K2):
                b: f32 = fifo_B[i, j].get()
        elif j == N2 + 1:
            for k in range(K2):
                a: f32 = fifo_A[i, j].get()
        else:
            c: f32 = 0
            for k in range(K2):
                a: f32 = fifo_A[i, j].get()
                b: f32 = fifo_B[i, j].get()
                c += a * b
                fifo_A[i, j + 1].put(a)
                fifo_B[i + 1, j].put(b)
            C[i - 1, j - 1] = c

    pe(A, B, C, fifo_A, fifo_B)

code_str = systolic_2d.schedule().export("vitis").hls_code
print(code_str)


# Each PE specialized into its own function; idle corner (0,0) pruned.
pe_funcs = sorted({ln.split("(")[0].split()[-1]
                   for ln in code_str.splitlines()
                   if ln.startswith("void systolic_2d_pe_")})
print("specialized PE functions:", pe_funcs)
print("corner PE 0_0 pruned:", "systolic_2d_pe_0_0" not in code_str)
print("interior PE present  :", "systolic_2d_pe_1_1(" in code_str)

The streams lower to `hls::stream` FIFOs and the whole array runs as an HLS **dataflow**
region — the PEs execute concurrently, communicating only through the FIFOs. This is the
dataflow architecture the lecture argued FPGAs are the natural home for.

## 4.3 Interfaces for deployment (AXI)

For a real accelerator the top-level arrays become AXI ports: an `m_axi` master for bulk
DRAM data. `set_axi(arg_index, offset=..., bundle=...)` records this on the backend, and
it appears as an `INTERFACE` pragma in the generated code.

In [ ]:
@kernel
def axicopy(A: i32[64], B: i32[64]):
    for i in allo.range(64, name="i"):
        B[i] = A[i] + 1

backend = axicopy.schedule().export("vitis", part=PART)
backend.set_axi(0, offset="slave", bundle="gmem")
backend.set_axi(1, offset="slave", bundle="gmem")
for ln in backend.hls_code.splitlines():
    if "INTERFACE" in ln:
        print(ln.strip())

## 4.4 Synthesis and the report

`export("vitis", part=..., project_path=...).synth()` invokes **Vitis HLS C-synthesis**
and returns a `VitisSynthReport`. Use `report.render()` for a formatted summary, or read
`report.xml_path` for the raw `csynth.xml`.

> ⚠️ This runs the real tool — expect a few minutes. Gated on `is_vitis_available()`.

In [ ]:
def synthesize(kernel_or_schedule, part=PART):
    s = kernel_or_schedule
    proj = tempfile.mkdtemp()                # kept so the report files survive
    report = s.export("vitis", part=part, project_path=proj).synth()
    return report

if is_vitis_available():
    s = systolic_2d.schedule()
    report = synthesize(s)
else:
    print("Vitis HLS not available -- skipping synthesis.")

### Pulling numbers out programmatically

For sweeps or plots it is handy to parse the report yourself. The `csynth.xml` schema
mirrors the report tables:

* `./PerformanceEstimates/SummaryOfOverallLatency/Average-caseLatency`
* `./AreaEstimates/Resources/{DSP,LUT,FF,BRAM_18K,URAM}`

In [ ]:
def qor(report):
    root = ET.parse(report.xml_path).getroot()
    lat = root.find("./PerformanceEstimates/SummaryOfOverallLatency")
    res = root.find("./AreaEstimates/Resources")
    def g(node, tag):
        el = node.find(tag) if node is not None else None
        return el.text if el is not None else "-"
    return {
        "latency(cyc)": g(lat, "Average-caseLatency"),
        "II": g(lat, "PipelineInitiationInterval"),
        "DSP": g(res, "DSP"), "LUT": g(res, "LUT"),
        "FF": g(res, "FF"), "BRAM": g(res, "BRAM_18K"),
    }

if is_vitis_available():
    print("systolic 2x2 GEMM QoR:", qor(report))
else:
    print("Vitis HLS not available.")

## 4.5 Naive vs systolic — a QoR comparison

The same matrix multiply, two ways: a plain pipelined triple loop versus the spatial
systolic array. Synthesizing both shows the trade the lecture described — the systolic
design trades more on-chip parallel hardware for higher throughput / lower latency, which
is exactly the structure FPGAs reward.

> Two synthesis runs — slow. Skip if you have no toolchain.

In [ ]:
def make_naive_gemm(sz):
    @kernel
    def gemm(A: f32[sz, sz], B: f32[sz, sz], C: f32[sz, sz]):
        for i in allo.range(sz, name="i"):
            for j in allo.range(sz, name="j"):
                for k in allo.range(sz, name="k"):
                    C[i, j] += A[i, k] * B[k, j]
    s = gemm.schedule()
    s.reorder((s.loop("i"), s.loop("k"), s.loop("j")))
    s.pipeline(s.loop("j"), ii=1)
    return s

if is_vitis_available():
    naive_report = synthesize(make_naive_gemm(2))     # match the 2x2 systolic size
    print(f"{'design':<14}{'latency':>10}{'DSP':>6}{'LUT':>8}{'FF':>8}")
    for name, rep in [("naive 2x2", naive_report), ("systolic 2x2", report)]:
        q = qor(rep)
        print(f"{name:<14}{str(q['latency(cyc)']):>10}{str(q['DSP']):>6}"
              f"{str(q['LUT']):>8}{str(q['FF']):>8}")
else:
    print("Vitis HLS not available -- skipping the comparison.")

## Where to go next

The `test/` directory has more spatial designs you can synthesize the same way — all
built on the SPMD `mapping` + `Stream` model:

* `test_systolic_gemm.py` — the 2D array above, plus a 1D (row) variant.
* `test_systolic_tiled.py` — a `Mt × Nt` PE array that sweeps over output tiles
  (decoupling array size from matrix size).
* `test_systolic_osws.py` — output-stationary / weight-stationary variants.
* `test_systolic_conv.py`, `test_systolic_smith_waterman.py` — convolution and a
  sequence-alignment systolic array.

## Course wrap-up

Across these four notebooks we have followed the lecture's thesis all the way down to
silicon:

* **Payload IR vs Transform IR** — the algorithm is written once; *How* is a separate,
  first-class object (notebooks 1–2).
* **Composable, non-destructive** schedules replace destructive pragmas (notebook 2).
* **Bit-accurate simulation** verifies correctness cheaply before synthesis (notebook 3).
* **Spatial / dataflow** designs — the systolic GEMM — that are awkward in plain HLS fall
  out naturally from an explicit spatial schedule, and synthesize to real FPGA hardware
  (notebook 4).

The open question the lecture left you with stands: when the schedule space is huge and
the constraints (timing!) are hard, do we explore it with cost models, with search, or
with learned policies? Allo gives you the *substrate* — an explicit, composable,
searchable space of hardware schedules — to go and answer it.